In [2]:
import os
import time
import json
import pickle
import pandas as pd
import numpy as np

import re
from bs4 import BeautifulSoup

from datetime import datetime
from tqdm import tqdm
from pathlib import Path

Define path variables

In [3]:
current_dir = Path.cwd()

train_data_path = current_dir.parent.parent / "data" / "01_processed" / "pan20-authorship-verification-training-large.jsonl"
train_data_truth_path = current_dir.parent.parent / "data" / "01_processed" / "pan20-authorship-verification-training-large-truth.jsonl"

test_data_path = current_dir.parent.parent / "data/01_processed/pan21-authorship-verification-test.jsonl"
test_data_truth_path = current_dir.parent.parent / "data/01_processed/pan21-authorship-verification-test-truth.jsonl"

train_data_full_cleaned_path = current_dir.parent.parent / "data" / "01_processed" / "pan20-authorship-verification-training-large-cleaned.jsonl"
test_data_full_cleaned_path = current_dir.parent.parent / "data/01_processed/pan21-authorship-verification-test-cleaned.jsonl"

# Load dataset

## Load training dataset

In [4]:
train_data_file_size = os.path.getsize(train_data_path)
train_data = []

with open(train_data_path, 'r') as f:
    with tqdm(total=train_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            train_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(train_data)} items.")

Loading data: 100%|██████████| 12.0G/12.0G [02:28<00:00, 80.8MB/s]


Successfully loaded 275565 items.


Load the data into the pandas dataframe

In [10]:
train_data_df = pd.DataFrame(train_data)
train_data_df.drop(columns=['fandoms'], inplace=True)
print(train_data_df.head(10))

                                     id  \
0  6b177179-72d2-5c87-ba2e-7678ae8c1db2   
1  80ff51a1-8f2b-507e-bece-e0a187f26a19   
2  b6492f44-4d7b-51d2-a6c8-fdfaf62868e4   
3  c502df34-8c2e-5a86-9555-cdcf0b03f189   
4  653cfc2d-5e82-5afa-87b1-c5b793441483   
5  7284fa07-32de-5ab8-a5b3-5b040bb59898   
6  ec4ca49c-5975-5b25-8f9b-bfc774775ce2   
7  7b312e06-d0d3-503b-ab52-997f99dd29bb   
8  2618e80d-1150-5d31-90ff-818d28e40724   
9  531e63a7-5db2-5ba8-9737-ad2ceab0265a   

                                                pair  
0  ["Alright..." We looked at the crowd of wide-e...  
1  [I had a rude awakening when a goblin threw so...  
2  ["The offer still stands." She stopped breathi...  
3  ["Yeah." "Just pretend you"ve put on the recor...  
4  ["Paloma just went "aww" like it could actuall...  
5  ["Good, honey, how are you?" "Oh, I"m... Deali...  
6  ["Something wrong?" he ventures timidly. "I......  
7  [Down, down they tumble down the hole, filled ...  
8  ["Well, why don"t you hold o

In [7]:
train_data_truth_file_size = os.path.getsize(train_data_truth_path)
train_data_truth = []

with open(train_data_truth_path, 'r') as f:
    with tqdm(total=train_data_truth_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            train_data_truth.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(train_data_truth)} items.")

Loading data: 100%|██████████| 26.4M/26.4M [00:02<00:00, 10.6MB/s]


Successfully loaded 275565 items.


Load the data into pandas dataframe

In [9]:
train_data_truth_df = pd.DataFrame(train_data_truth)
train_data_truth_df.drop(columns=['authors'], inplace=True)
print(train_data_truth_df.head())

                                     id  same
0  6b177179-72d2-5c87-ba2e-7678ae8c1db2  True
1  80ff51a1-8f2b-507e-bece-e0a187f26a19  True
2  b6492f44-4d7b-51d2-a6c8-fdfaf62868e4  True
3  c502df34-8c2e-5a86-9555-cdcf0b03f189  True
4  653cfc2d-5e82-5afa-87b1-c5b793441483  True


Combine the dataframes for evaluation

In [7]:
train_data_full_df = pd.merge(train_data_df, train_data_truth_df, on='id')
print(train_data_full_df.head())
train_data_full_df.head(5).to_csv("original5.csv")

                                     id  \
0  6b177179-72d2-5c87-ba2e-7678ae8c1db2   
1  80ff51a1-8f2b-507e-bece-e0a187f26a19   
2  b6492f44-4d7b-51d2-a6c8-fdfaf62868e4   
3  c502df34-8c2e-5a86-9555-cdcf0b03f189   
4  653cfc2d-5e82-5afa-87b1-c5b793441483   

                                                pair  same  
0  ["Alright..." We looked at the crowd of wide-e...  True  
1  [I had a rude awakening when a goblin threw so...  True  
2  ["The offer still stands." She stopped breathi...  True  
3  ["Yeah." "Just pretend you"ve put on the recor...  True  
4  ["Paloma just went "aww" like it could actuall...  True  


## Load testing dataset

In [8]:
test_data_file_size = os.path.getsize(test_data_path)
test_data = []

with open(test_data_path, 'r') as f:
    with tqdm(total=test_data_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            test_data.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(test_data)} items.")

Loading data: 100%|██████████| 873M/873M [00:09<00:00, 91.2MB/s] 


Successfully loaded 19999 items.


Load the data into the pandas dataframe

In [9]:
test_data_df = pd.DataFrame(test_data)
test_data_df.drop(columns=['fandoms'], inplace=True)
print(test_data_df.head())

                                     id  \
0  c28e8b03-c02a-5184-b58a-12dd28b8ca74   
1  b9326101-6352-56dd-9d1b-1f41466897b7   
2  e2ac4453-bf54-53f2-bf68-6caae6aacded   
3  a5e9a289-0999-5764-b597-dc1bf8c21ede   
4  cb4054b1-d422-58d6-a137-dcfc70100df6   

                                                pair  
0  [talk because they hadn"t been exposed to comm...  
1  [Zazuki nodded his head and got to his feet, k...  
2  ["Oh we did lots of special things. On Christm...  
3  ["Hey now, at least Shido brings home some mon...  
4  [It was a mere five minutes" walk from third y...  


Load the testing set labels

In [10]:
test_data_truth_file_size = os.path.getsize(test_data_truth_path)
test_data_truth = []

with open(test_data_truth_path, 'r') as f:
    with tqdm(total=test_data_truth_file_size, desc="Loading data", unit='B', unit_scale=True) as pbar:
        for line in f:
            test_data_truth.append(json.loads(line))
            pbar.update(len(line.encode('utf-8')))

print(f"\nSuccessfully loaded {len(test_data_truth)} items.")

Loading data: 100%|██████████| 1.91M/1.91M [00:00<00:00, 26.4MB/s]


Successfully loaded 19999 items.


Load the data into pandas dataframe

In [11]:
test_data_truth_df = pd.DataFrame(test_data_truth)
test_data_truth_df.drop(columns=['authors'], inplace=True)
print(test_data_truth_df.head())

                                     id   same
0  c28e8b03-c02a-5184-b58a-12dd28b8ca74   True
1  b9326101-6352-56dd-9d1b-1f41466897b7   True
2  e2ac4453-bf54-53f2-bf68-6caae6aacded  False
3  a5e9a289-0999-5764-b597-dc1bf8c21ede   True
4  cb4054b1-d422-58d6-a137-dcfc70100df6   True


Combine the dataframes for evaluation

In [12]:
test_data_full_df = pd.merge(test_data_df, test_data_truth_df, on='id')
test_data_full_reduced_df =  test_data_full_df.head(1)
print(test_data_full_df.head())

                                     id  \
0  c28e8b03-c02a-5184-b58a-12dd28b8ca74   
1  b9326101-6352-56dd-9d1b-1f41466897b7   
2  e2ac4453-bf54-53f2-bf68-6caae6aacded   
3  a5e9a289-0999-5764-b597-dc1bf8c21ede   
4  cb4054b1-d422-58d6-a137-dcfc70100df6   

                                                pair   same  
0  [talk because they hadn"t been exposed to comm...   True  
1  [Zazuki nodded his head and got to his feet, k...   True  
2  ["Oh we did lots of special things. On Christm...  False  
3  ["Hey now, at least Shido brings home some mon...   True  
4  [It was a mere five minutes" walk from third y...   True  


 # Preprocess text

In [13]:
def clean_text(text):
    soup = BeautifulSoup(text, "html.parser")
    text = re.sub('\[[^]]*\]', '', soup.get_text())
    pattern=r"[^a-zA-z0-9\s,']"
    text=re.sub(pattern,'',text)
    return text

In [14]:
tqdm.pandas()

Preprocess test dataset

In [15]:
test_data_full_df['pair'] = test_data_full_df['pair'].progress_apply(lambda lst: [clean_text(x) for x in lst])

100%|██████████| 19999/19999 [00:17<00:00, 1118.35it/s]


Preprocess train dataset

In [16]:
train_data_full_df['pair'] = train_data_full_df['pair'].progress_apply(lambda lst: [clean_text(x) for x in lst])

100%|██████████| 275801/275801 [04:12<00:00, 1093.45it/s]


# Export 

Export test dataset

In [17]:
with open(test_data_full_cleaned_path, "w", encoding="utf-8") as f:
    for _, row in test_data_full_df.iterrows():
        obj = {
            "id" : row["id"],
            "pair": row["pair"],      
            "same": row["same"]
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

Export train dataset

In [18]:
with open(train_data_full_cleaned_path, "w", encoding="utf-8") as f:
    for _, row in train_data_full_df.iterrows():
        obj = {
            "id" : row["id"],
            "pair": row["pair"],      
            "same": row["same"]
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")